# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Cite as: {getattr(metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The record sets organize the available tables of data. Each record set exposes fields (columns), each with its own `@id`. This ensures consistent references across the notebook.

In [ ]:
# Fetch the list of record set IDs
record_sets = []
for rs in metadata.record_sets:
    print(f"Record set name: {rs.name}")
    print(f"Record set @id: {rs.id}")
    field_ids = [f.id for f in rs.fields]
    print(f"Fields (@id): {field_ids}")
    record_sets.append(rs.id)
    print('---')
    # Preview a sample record using the 'records' generator
    try:
        record_iter = dataset.records(record_set=rs.id)
        sample = next(record_iter)
        print("Sample record:")
        print(json.dumps(sample, indent=2))
    except Exception as e:
        print(f"Could not print sample record: {e}")
    print("======================================\n")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame using their `@id`s for further processing and analysis.

In [ ]:
# Extract all record sets into separate dataframes
dataframes = {}

for rs_id in record_sets:
    print(f"Loading records for record set '@id': {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for {rs_id}.")

## 4. Exploratory Data Analysis (EDA)
Demonstrate data filtering, normalization, and grouping using a numeric field. 

Replace the variables below (`target_record_set_id`, `numeric_field_id`, `group_field_id`) with the actual `@id`s of a relevant record set and field for your real analysis, as found above.

In [ ]:
# Select a record set and fields to analyze
# Replace with actual values from the overview if different
target_record_set_id = record_sets[0]  # e.g., first record set
# Choose a numeric field '@id', e.g., 'cr:gad_score'
# Choose a categorical/group field '@id', e.g., 'cr:gender'

# For demonstration, print all columns to aid selection
print(f"Available columns in {target_record_set_id}:")
print(dataframes[target_record_set_id].columns.tolist())

numeric_field_id = None
group_field_id = None
# Try to automatically select common mental health scores as numeric fields
possible_numeric_fields = [
    col for col in dataframes[target_record_set_id].columns 
    if any(x in col.lower() for x in ['score', 'phq', 'gad', 'psq'])
]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No common numeric score field found.")
    numeric_field_id = dataframes[target_record_set_id].select_dtypes(include='number').columns[0]

# Try to get a group field (e.g., gender or age_group)
possible_group_fields = [col for col in dataframes[target_record_set_id].columns if col.lower() in ['gender', 'age_group', 'marital_status', 'village']]
if possible_group_fields:
    group_field_id = possible_group_fields[0]
    print(f"Using group field: {group_field_id}")

# Filter for high scores (outliers/interesting cases)
threshold = 10
if numeric_field_id:
    filtered_df = dataframes[target_record_set_id][dataframes[target_record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group field, if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print("No suitable numeric field identified for EDA.")

## 5. Visualization
Visualize the distribution of a numeric score and its relationship to an attribute such as gender if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(dataframes[target_record_set_id][numeric_field_id], kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group field
    if group_field_id and group_field_id in dataframes[target_record_set_id].columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[target_record_set_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load the FAIR² Kilifi Mental Health Survey dataset using `mlcroissant` by referencing entities with their `@id` values as per the Croissant schema. 

- **We loaded and explored available record sets and their field IDs**
- **Data were extracted into pandas DataFrames for each record set**
- **We performed basic exploratory analysis and visualized the results, using only standard Python, Pandas, Matplotlib, and Seaborn tools**

This process can serve as a foundation for further, in-depth statistical analysis and machine learning applications with FAIR and AI-ready datasets.